# Root-Aligned SMILES on the ORD-free base — variant 6 re-run on `sagawa/CompoundT5`

Retries the one variant whose recorded failure was explained by a property of the base
checkpoint rather than by the method. `RESULTS.md` on variant 6: root-aligned targets scored
40.7% ORD top-1 exact against variant 2's 50.3%, and the stated reason was that *the base
checkpoint is already pretrained to emit canonical SMILES, and 57,000 examples over 3 epochs
were not enough to retrain it onto a different (root) convention*.

`sagawa/CompoundT5` has no reaction-output prior at all — it has never emitted a reactant set in
either convention, so it learns whichever one it is shown from scratch. That makes this the first
test of Zhong et al. (Chemical Science, 2022) on this data where the base does not fight the
representation. Reported effect in the literature: up to +9.23 top-1 points from the data
representation alone.

**Matched control.** Identical to `12_train_reactant_compoundt5_57k.ipynb` in every respect —
same 57,000-reaction pool size, `--no-augment`, `lr=5e-4`, 3 epochs, `torchrun --nproc_per_node=2`
— with the reactant targets rewritten into root-aligned form as the only difference. Products
(the model input) are untouched canonical SMILES, so inference needs no changes and downstream
canonicalization in the eval script handles the non-canonical output. Compare against:

| | ORD exact top-1 | ORD exact top-5 | ORD core top-5 |
|---|---|---|---|
| CompoundT5 + 57k canonical | 17.0% | 23.7% | 34.0% |
| CompoundT5 + 147k canonical | 19.7% | 28.3% | 39.7% |
| ReactionT5 variant 2 (57k canonical) | 50.3% | 73.0% | 79.7% |
| ReactionT5 variant 6 (57k root-aligned) | 40.7% | 59.3% | 69.0% |

**Data:** `kuzmenkooleh/retro-planner-ord-rootaligned-57k`, built by
`scripts/build_root_aligned_data.py` from the same eval-excluded 57k pool as variant 2 (99.6%
successfully root-aligned via RXNMapper + RDKit `rootedAtAtom`; 234 fell back to plain canonical
on mapping failures and were kept, so pool size is unchanged).

~1 h 31 min training plus ~15 min for the two evaluations.

**Before running:** Settings -> **Internet** on, **GPU T4 x2** on. Run as **Save & Run All
(Commit)**.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  Device {i}:", torch.cuda.get_device_name(i))

In [ ]:
import os
if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

In [ ]:
import glob

# Kaggle has mounted datasets under two different layouts historically, so search
# rather than hard-code the path.
train_file = next(glob.iglob("/kaggle/input/**/reactants_train.jsonl", recursive=True))
val_file = next(glob.iglob("/kaggle/input/**/reactants_val.jsonl", recursive=True))
for path in (train_file, val_file):
    print(path, sum(1 for _ in open(path)), "rows")

base_model = "sagawa/CompoundT5"
learning_rate = 5e-4  # matched to the 57k canonical run, so representation is the only difference
output_dir = "/kaggle/working/model1_compoundt5_rootaligned57k"
time_budget_minutes = 200  # ~91 min training + headroom

In [ ]:
# Confirms the mounted dataset really is the root-aligned pool and not the canonical 57k:
# both carry identical filenames, and the glob above would silently pick either.
import json

with open(train_file) as handle:
    for _ in range(3):
        row = json.loads(handle.readline())
        print("product :", row["product_smiles"][:80])
        print("reactant:", row["reactants_smiles"][:80])

In [ ]:
import os

os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"

!torchrun --nproc_per_node=2 scripts/train_reactant_model_ord.py \
    --base-model "{base_model}" \
    --train-file "{train_file}" \
    --val-file "{val_file}" \
    --output-dir "{output_dir}" \
    --local-work-dir /kaggle/temp/local_model1_work \
    --no-augment \
    --learning-rate {learning_rate} \
    --num-train-epochs 3 \
    --time-budget-minutes {time_budget_minutes} \
    > "{log_path}" 2>&1
print("training done; tail of log:")
!tail -5 "{log_path}"

In [ ]:
# The repair must have fired. If this line is absent the run trained on <unk>-corrupted
# targets and its numbers are meaningless.
!grep -E "new character token|Train examples" "{log_path}"

import json
from transformers import AutoTokenizer

saved_tokenizer = AutoTokenizer.from_pretrained(f"{output_dir}/final")
embedding_rows = json.load(open(f"{output_dir}/final/config.json"))["vocab_size"]
print("tokenizer length:", len(saved_tokenizer), "| embedding rows:", embedding_rows)
assert len(saved_tokenizer) == embedding_rows, "tokenizer and embedding matrix disagree"

state = json.load(open(f"{output_dir}/latest_checkpoint/trainer_state.json"))
points = [(h["epoch"], h["eval_loss"]) for h in state["log_history"] if "eval_loss" in h]
for epoch, loss in points[::max(1, len(points) // 10)]:
    print(f"  epoch {epoch:5.2f}  eval_loss {loss:.4f}")
print("  last:", points[-1], "| best:", state.get("best_metric"))

In [ ]:
import os
model_dir = f"{output_dir}/final"
assert os.path.isdir(model_dir), os.listdir(output_dir)

for tag, targets in [("ord", "data/v2_ord_eval_targets.json"),
                     ("uspto", "data/v2_uspto_eval_targets.json")]:
    !python scripts/models/run_reactiont5_topk.py \
        --input "{targets}" --t5-model "{model_dir}" \
        --num-beams 10 --device cuda \
        --output "/kaggle/working/compoundt5_rootaligned57k_{tag}_topk.json"
    print(tag, "done")

In [ ]:
import json
for tag in ("ord", "uspto"):
    data = json.load(open(f"/kaggle/working/compoundt5_rootaligned57k_{tag}_topk.json"))
    print("===", tag, "===")
    print(json.dumps(data["summary"], indent=2))